<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session07-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 7 lab: linear regression vs. a small MLP, on a real biological target {.unnumbered}

A teaser, not a full machine-learning treatment: Session 8 covers the
fundamentals (train/validation/test splits, evaluation) properly, and
Session 9 covers regularization and architecture choices. Here, just:
assemble one small real dataset, fit two models on it the way the Session
6 page's "Learning the weights from examples" and "Why a hidden layer
helps" sections did on toy data, compare them, find out how much (or how
little) one small comparison can tell you, and then ask what the input
itself leaves out.

**The task**: predict a protein's alpha-helix fraction from its
amino-acid composition: the "composition view" of the page's HBB
figure, where beta-globin becomes 20 numbers and one target (0.80 of its
residues are in helices). Here the same PDBe API
is used for 55 structurally diverse proteins (a real range of folds, from
almost no helix to mostly helix): 20 features per protein (the fraction
of each amino acid in its sequence) and one target (its helix fraction).

## 1. Assemble the real dataset

55 PDB entries, picked by a resolution- and length-filtered search of
the RCSB and fixed here as a list so the cell runs the same way every
time. For each: fetch its sequence and secondary-structure assignment
from EBI's PDBe API (the same endpoints as the page's HBB example), then
compute its amino-acid composition and helix fraction.

If the PDBe API is not reachable (a network problem during the lab),
the cell uses a cached copy of exactly these values, made from the same
live fetch.

In [ ]:
import os
import requests
import numpy as np
import pandas as pd

PDB_IDS = ['21XG', '38LJ', '9WRU', '9NVT', '9XZ9', '7ILS', '9RYL', '9TLR', '32TC', '13DZ', '9TCX', '9YBX', '9N2Y', '29OL', '9PNX', '9YMO', '9HWE', '9RPQ', '9RR6', '9W2A', '21LO', '9RK7', '9OU3', '9P39', '9T4I', '9VOM', '9W28', '9RFG', '9V5C', '9RQP', '9I4O', '9QWL', '9QYO', '9WM8', '9QRF', '9X6R', '9TZG', '9QGV', '9U97', '9YK5', '9IC9', '9XPD', '9DJF', '9DJR', '9NK4', '9NCE', '9I1B', '9RQQ', '7IPN', '9LCS', '9Z70', '9KZN', '9SI4', '9P45', '9FD0']
AAS = list("ACDEFGHIKLMNPQRSTVWY")
CACHE = "session07-helix-dataset.csv"
CACHE_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/" + CACHE

def fetch_protein(pdbid):
    '''Sequence, chain, chain length and number of helix residues, live from PDBe.'''
    lo = pdbid.lower()
    r = requests.get(f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/molecules/{lo}", timeout=15)
    r.raise_for_status()
    mol = [m for m in r.json()[lo] if m.get("molecule_type") == "polypeptide(L)" and m.get("sequence")][0]
    seq, chain_id, length = mol["sequence"], mol["in_chains"][0], mol["length"]

    r = requests.get(f"https://www.ebi.ac.uk/pdbe/api/pdb/entry/secondary_structure/{lo}", timeout=15)
    r.raise_for_status()
    helices = []
    for molecule in r.json()[lo]["molecules"]:
        for chain in molecule["chains"]:
            if chain["chain_id"] == chain_id:
                helices = chain["secondary_structure"].get("helices", [])
    helix_residues = sum(h["end"]["residue_number"] - h["start"]["residue_number"] + 1 for h in helices)
    return {"pdb_id": pdbid, "chain": chain_id, "length": length,
            "helix_residues": helix_residues, "sequence": seq}

def load_cache():
    path = os.path.join("data", CACHE)
    return pd.read_csv(path if os.path.exists(path) else CACHE_URL, keep_default_na=False)

try:
    data = pd.DataFrame([fetch_protein(p) for p in PDB_IDS])
    source = "live from the PDBe API"
    try:                                          # the live values should match the cached copy
        cached = load_cache()[data.columns]
        print("live values match the cached copy:", bool((data.astype(str).values == cached.astype(str).values).all()))
    except Exception:
        pass
except Exception as err:
    print("PDBe API not reachable:", err)
    data = load_cache()
    source = "the cached copy (same values, fetched earlier)"

X = np.array([[seq.count(aa) / len(seq) for aa in AAS] for seq in data["sequence"]])
y = (data["helix_residues"] / data["length"]).to_numpy()

print(f"Dataset ({source}): {X.shape[0]} proteins, {X.shape[1]} features (amino-acid composition)")
print(f"Helix fraction: min={y.min():.2f}, max={y.max():.2f}, mean={y.mean():.2f}")
print(f"Sequence lengths: min={data['length'].min()}, max={data['length'].max()}")

## 2. Holding out data to test on

Testing on data not used for fitting gives an estimate of how well the
model generalizes to new examples. Fit both models on 75% of the
proteins and test both on the same 25% they never saw.

Keep in mind how rough this estimate is. One 25% holdout of 55 proteins
is only 14 test proteins, so the result depends a lot on which proteins
happen to land in the test set (Section 7 shows how much). And if
related proteins end up in both training and test sets, the estimate
can overstate how well the model works on genuinely new proteins.
Session 8 handles both problems properly.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(f"Training on {len(X_train)} proteins, testing on {len(X_test)} proteins not used for fitting.")

## 3. Linear regression

Linear regression has a direct numerical solution: scikit-learn computes
the best weights in one step with a least-squares solver. Gradient
descent would reach the same optimum iteratively, as the page's toy
example showed. (For the curious: the textbook formula is
$\hat{\mathbf{w}} = (X^\top X)^{-1} X^\top \mathbf{y}$, but good software
does not literally invert $X^\top X$.)

One detail of these particular features: the 20 amino-acid fractions of
a protein add up to 1 (for 54 of the 55 proteins; one sequence also
contains unknown residues, `X`). Together with the intercept, the
features are therefore almost exactly linearly dependent. The fit still
works, but the individual weights are poorly determined: many different
weight combinations give nearly the same predictions. Don't read meaning
into a single weight here.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

print("composition rows that sum to 1:", int(np.sum(np.isclose(X.sum(axis=1), 1.0))), "of", len(X))

linreg = LinearRegression().fit(X_train, y_train)
pred_linreg = linreg.predict(X_test)
mse_linreg = mean_squared_error(y_test, pred_linreg)

print(f"Linear regression: held-out MSE = {mse_linreg:.4f}")

## 4. A small multilayer perceptron, from scratch

The same raw-tensors, plain-gradient-descent style as the XOR notebook's
hidden-layer model: one small hidden layer (8 units, `tanh`), trained by
hand with no `nn.Module`, no optimizer class and no regularization
(Session 9). Only the loss changes: mean squared error for a number,
not binary cross-entropy for a yes/no class.

In [ ]:
import torch

def train_mlp(X_train, y_train, X_test, y_test, hidden=8, lr_rate=0.05, epochs=3000, seed=0):
    '''Train the one-hidden-layer MLP; return (held-out MSE, training losses).'''
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    Xte = torch.tensor(X_test, dtype=torch.float32)
    yte = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

    torch.manual_seed(seed)
    W1 = (torch.randn(Xtr.shape[1], hidden) * 0.3).requires_grad_()
    b1 = torch.zeros(hidden, requires_grad=True)
    W2 = (torch.randn(hidden, 1) * 0.3).requires_grad_()
    b2 = torch.zeros(1, requires_grad=True)
    params = [W1, b1, W2, b2]

    losses = []
    for epoch in range(epochs):
        hid = torch.tanh(Xtr @ W1 + b1)
        pred = hid @ W2 + b2
        loss = torch.nn.functional.mse_loss(pred, ytr)
        loss.backward()
        with torch.no_grad():
            for p in params:
                p -= lr_rate * p.grad
                p.grad.zero_()
        losses.append(loss.item())

    with torch.no_grad():
        pred_test = torch.tanh(Xte @ W1 + b1) @ W2 + b2
        return torch.nn.functional.mse_loss(pred_test, yte).item(), losses

mse_mlp, epoch_losses = train_mlp(X_train, y_train, X_test, y_test)
print(f"MLP (8-unit hidden layer): held-out MSE = {mse_mlp:.4f}")
print(f"Training MSE: {epoch_losses[0]:.4f} (epoch 0) -> {epoch_losses[-1]:.4f} (epoch 3000)")

## 5. The training curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(epoch_losses)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.title("MLP training loss (held-out MSE = {:.4f})".format(mse_mlp))
plt.tight_layout()
plt.show()

## 6. So which one predicted better?

Compare the two held-out errors, and look at the size of the difference
before calling a winner.

In [ ]:
print(f"Linear regression held-out MSE: {mse_linreg:.4f}")
print(f"MLP (8-unit hidden layer) held-out MSE: {mse_mlp:.4f}")
print(f"Difference: {abs(mse_linreg - mse_mlp):.4f}")

The two errors differ by about 0.0001. On 14 test proteins that is
**essentially a tie on this split**, not evidence that either model is
better. Don't overinterpret tiny differences: the next section shows how
much the result changes when only the split changes.

## 7. How noisy is one split?

Repeat the whole comparison on 10 different random splits (the same
data, only a different choice of test proteins). The optional last
section uses the `linear6` column.

In [ ]:
subset = ["A", "E", "L", "M", "P", "G"]         # helix formers and breakers (used in the optional section)
idx = [AAS.index(aa) for aa in subset]

rows = []
for seed in range(10):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=seed)
    lin = mean_squared_error(yb, LinearRegression().fit(Xa, ya).predict(Xb))
    mlp, _ = train_mlp(Xa, ya, Xb, yb)
    lin6 = mean_squared_error(yb, LinearRegression().fit(Xa[:, idx], ya).predict(Xb[:, idx]))
    rows.append({"split": seed, "linear": lin, "mlp": mlp, "linear6": lin6})
splits = pd.DataFrame(rows)
print(splits.round(4).to_string(index=False))
print(f"\nlinear regression test MSE ranges from {splits['linear'].min():.3f} to {splits['linear'].max():.3f}")
print(f"the MLP has the lower test MSE in {(splits['mlp'] < splits['linear']).sum()} of 10 splits")

The test error changes far more from one split to the next than the
0.0001 between the two models on split 0, and the "winner" changes
too. This small dataset gives no evidence that either model is better.

## 8. What has the input thrown away?

A hidden layer can model many kinds of nonlinear relationships, not only
XOR-like interactions, including a nonlinear dependence on a single
feature. So "a linear model is enough" is not the lesson. Before reading
on, stop and answer this in your group (the Canvas quiz asks it too):

> **If we really wanted to predict the secondary structure of each
> residue, what important information have we thrown away by
> representing an entire protein by 20 composition numbers?**

Write your answer in the cell below before you run the next code cell.

*Your answer:*

**Check your answer.** Composition keeps only how often each amino acid
occurs. It throws away the order of the residues, and with it each
residue's local context (its neighbours) and its position in the chain.
Two proteins with the same composition but completely different
arrangements of residues have *identical* input vectors:

In [ ]:
rng = np.random.default_rng(0)
seq = data.loc[np.argmax(y), "sequence"]         # the most helical protein in the dataset
scrambled = "".join(rng.permutation(list(seq)))
comp = lambda s: np.array([s.count(aa) / len(s) for aa in AAS])
print("original :", seq[:50], "...")
print("scrambled:", scrambled[:50], "...")
print("identical composition vectors:", np.allclose(comp(seq), comp(scrambled)))

The scrambled sequence would not fold into the same helices, yet every
model trained on composition must give it exactly the same prediction.
No model, linear or not, can recover information that is absent from
its inputs: a more powerful model cannot compensate for an impoverished
representation. Better predictions need better inputs. This is exactly
what the page's secondary structure predictors did: Qian and Sejnowski
and PHD read a *window* of neighbouring residues around each position
(order and local context), and PHD's step up came from replacing the
single sequence in that window with a *profile* of its homologs. Later
sessions use
models that read the sequence itself and learn their own
representations of it: convolutional networks (Session 10) and protein
language models (Session 11).

## Optional: does every feature help?

Try a smaller subset of the 20 features: just the six amino acids
classically associated with helix formation (alanine, glutamate,
leucine, methionine) and breaking (proline, glycine).

In [ ]:
X_train_sub = X_train[:, idx]
X_test_sub = X_test[:, idx]

linreg_sub = LinearRegression().fit(X_train_sub, y_train)
mse_sub = mean_squared_error(y_test, linreg_sub.predict(X_test_sub))

print(f"Linear regression, all 20 features:      held-out MSE = {mse_linreg:.4f}")
print(f"Linear regression, these 6 features only: held-out MSE = {mse_sub:.4f}")
print(f"\nacross the 10 splits of Section 7, the six-feature model has the lower test MSE in "
      f"{(splits['linear6'] < splits['linear']).sum()} of 10")

On this split, the six-feature model happens to give slightly lower test
error. **Why isn't that enough to conclude that six features are
generally better?** (One split of 14 proteins; Section 7 showed how much
that varies.) Across the 10 splits the six-feature model does better
more often, which is stronger evidence, but the splits share most of
their proteins, so they are not independent. Whether a simpler model
really generalizes better, and why fitting 20 weights to about 40
proteins can hurt, is exactly what Sessions 8 (cross-validation) and 9
(overfitting and regularization) are about.

## Done

Once every cell above has run and printed output, answer the Session 7
lab quiz on Canvas using your own numbers.